# Environment Setup

In [1]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 160.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 275.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
%%capture
%pip install -U bitsandbytes
%pip install -U peft
%pip install -U accelerate
%pip install -U trl
%pip install -U datasets

In [3]:
%pip install datasets==2.16.0

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.21.0 requires datase

# Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from random import randrange
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments,BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import DPOTrainer, DPOConfig
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

## Wandb

In [ ]:
import wandb

# Log in to your W&B account
wandb.login()

# Initialize a new W&B run
wandb.init(
    project="dpo-llama3.1-8b-instruct-cloud-zero-with-ocr-qa"
)

wandb: Currently logged in as: abrahamkaikobadtushar (axiler) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Model & Tokenizer Loading

In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
#torch.cuda.empty_cache()
#torch.cuda.reset_peak_memory_stats()

In [5]:
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login
import os

token = os.getenv('HF_TOKEN')
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name= "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto",use_cache=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
model_name= "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:
# ref model
#ref_model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto")

# Dataset load and preprocess

In [11]:
import pandas as pd
cloud_train=pd.read_csv("../../data/splits/cloud/dpo/dpo_train_cloud_data_qa.csv")

In [39]:
cloud_train.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt,choosen_explanation_responses,rejected_limitations_responses
0,OCR Text:\nimg0:\nNon-table text:\n ...,A service connection is a connector to an exte...,The Docker registry service connection is used...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",The chosen answer is correct because it explai...,**Limitations and flaws in the rejected answer...
1,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYour error cognito-idp:ListUsers is about ...,The issue here seems that your role does not a...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",The chosen answer is correct because it identi...,Certainly! Here’s an explanation of why the re...


In [40]:
cloud_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2847 entries, 0 to 2846
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   question                        2847 non-null   object
 1   chosen                          2847 non-null   object
 2   rejected                        2847 non-null   object
 3   choosen_explanation_prompt      2847 non-null   object
 4   rejected_limitations_prompt     2847 non-null   object
 5   choosen_explanation_responses   2847 non-null   object
 6   rejected_limitations_responses  2846 non-null   object
dtypes: object(7)
memory usage: 155.8+ KB


## processed

In [41]:
cloud_train=cloud_train[['question','chosen','choosen_explanation_responses','rejected','rejected_limitations_responses']]

In [42]:
cloud_train.head(2)

,question,chosen,choosen_explanation_responses,rejected,rejected_limitations_responses
0,OCR Text:\nimg0:\nNon-table text:\n ...,A service connection is a connector to an exte...,The chosen answer is correct because it explai...,The Docker registry service connection is used...,**Limitations and flaws in the rejected answer...
1,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYour error cognito-idp:ListUsers is about ...,The chosen answer is correct because it identi...,The issue here seems that your role does not a...,Certainly! Here’s an explanation of why the re...


In [43]:
cloud_train['chosen'] = (
    "chosen-answer: " + cloud_train['chosen'].astype(str).str.strip() + 
    "\nexplanation: " + cloud_train['choosen_explanation_responses'].astype(str).str.strip()
)

In [44]:
cloud_train['rejected'] = (
    "rejected-answer: " + cloud_train['rejected'].astype(str).str.strip() + 
    "\nlimitations: " + cloud_train['rejected_limitations_responses'].astype(str).str.strip()
)

In [45]:
cloud_train_processed=pd.DataFrame(
    {
        'prompt':cloud_train['question'],
        'chosen':cloud_train['chosen'],
        'rejected':cloud_train['rejected']
    }
)

In [46]:
cloud_train_processed.head(2)

,prompt,chosen,rejected
0,OCR Text:\nimg0:\nNon-table text:\n ...,chosen-answer: A service connection is a conne...,rejected-answer: The Docker registry service c...
1,OCR Text:\nimg0:\nNon-table text:\n ...,chosen-answer: Your error cognito-idp:ListUser...,rejected-answer: The issue here seems that you...


## prompt 

In [47]:
zero_shot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

{}

Answer:

'''

In [48]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['prompt'])

cloud_train_processed['prompt'] = cloud_train_processed.apply(generate_zero_shot_prompt, axis=1)


In [49]:
print(cloud_train_processed['prompt'][1])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
                                                                                                                                                                                                                    Import managed policy
      Visual editor
                              JSON
    Expand all Collapse all
          Cognito Identity (1 action)
                                                                                                                                                                                                                       Clone Remove
                                                ▸ Service Cognito Identity
                                                ► Actions
                                      

## dataset format

In [50]:
# 20% validation set
val_df = cloud_train_processed.sample(frac=0.2, random_state=42)

# 80% training set 
train_df = cloud_train_processed.drop(val_df.index)

In [51]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
#train_df=train_df[:300]
#val_df=val_df[:100]

In [53]:
from dotenv import load_dotenv
load_dotenv()
from datasets import Dataset
cloud_dpo_trainset = Dataset.from_pandas(train_df[["prompt", "chosen", "rejected"]])
cloud_dpo_valset = Dataset.from_pandas(val_df[["prompt", "chosen", "rejected"]])

In [54]:
cloud_dpo_trainset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [55]:
cloud_dpo_valset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 100
})

# Training Arguments

In [56]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from trl import DPOConfig

training_args = DPOConfig(
    output_dir="dpo-llama3.1-8b-instruct-cloud-zero-with-ocr-qa",
    num_train_epochs=5,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    gradient_checkpointing=True,
    seed=42,
    push_to_hub=True,
    load_best_model_at_end=True,
    hub_model_id="Bakugo123/dpo-llama3.1-8b-instruct-cloud-zero-with-ocr-qa",
    hub_strategy="end",
    beta=0.1  
)

# Trainer

In [59]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [60]:
dpo_trainer = DPOTrainer(
    model=model,
    #ref_model=ref_model,
    args=training_args,
    train_dataset=cloud_dpo_trainset,
    eval_dataset=cloud_dpo_valset,
    processing_class=tokenizer,
    peft_config=peft_config
)


/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Extracting prompt in train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [61]:
# train the model
dpo_trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
10,No log,0.461399,0.063307,-0.481109,1.000000,0.544416,-657.166138,-978.200256,-0.484574,-0.473397
20,No log,0.043051,0.082735,-3.533957,1.000000,3.616693,-656.971924,-1008.728760,-0.475102,-0.451548
30,No log,0.011294,-0.225779,-7.343657,1.000000,7.117879,-660.057068,-1046.825806,-0.465490,-0.433766
40,No log,0.008455,-0.381113,-9.230501,1.000000,8.849390,-661.610352,-1065.694214,-0.460856,-0.427600
50,0.176400,0.007554,-0.403021,-9.742044,1.000000,9.339024,-661.829407,-1070.809692,-0.458632,-0.424877
60,0.176400,0.008290,-0.406620,-9.855148,1.000000,9.448527,-661.865356,-1071.940674,-0.457953,-0.423894
70,0.176400,0.008123,-0.395939,-9.867471,1.000000,9.471533,-661.758667,-1072.063965,-0.458038,-0.423649


TrainOutput(global_step=78, training_loss=0.1132273559506314, metrics={'train_runtime': 1857.2419, 'train_samples_per_second': 0.969, 'train_steps_per_second': 0.042, 'total_flos': 0.0, 'train_loss': 0.1132273559506314, 'epoch': 6.0})

In [62]:
# Evaluate the model
eval_results = dpo_trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

Evaluation Results: {'eval_loss': 0.008222481235861778, 'eval_runtime': 41.3258, 'eval_samples_per_second': 2.42, 'eval_steps_per_second': 0.315, 'eval_rewards/chosen': -0.3977380096912384, 'eval_rewards/rejected': -9.872909545898438, 'eval_rewards/accuracies': 1.0, 'eval_rewards/margins': 9.475171089172363, 'eval_logps/chosen': -661.776611328125, 'eval_logps/rejected': -1072.1182861328125, 'eval_logits/chosen': -0.4577179253101349, 'eval_logits/rejected': -0.4237818419933319, 'epoch': 6.0}


In [ ]:
# Save the model and tokenizer
#dpo_trainer.save_model("dpo-llama3-8b-instruct-cloud-zero-with-ocr-qa")
#tokenizer.save_pretrained("dpo-llama3-8b-instruct-cloud-zero-with-ocr-qa")

# Save Fine Tune Model

In [ ]:
#model.push_to_hub("Bakugo123/sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")
#tokenizer.push_to_hub("Bakugo123/sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")

In [63]:
dpo_trainer.push_to_hub()

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...ero-with-ocr-qa-test/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...r-qa-test/adapter_model.safetensors:   1%|          |  560kB /  109MB            

  ...-with-ocr-qa-test/training_args.bin:  11%|#         |   728B / 6.80kB            

CommitInfo(commit_url='https://huggingface.co/Bakugo123/dpo-llama3-8b-instruct-cloud-zero-with-ocr-qa-test/commit/61394849b623e838e34af2dcebad0d94f79bd96e', commit_message='End of training', commit_description='', oid='61394849b623e838e34af2dcebad0d94f79bd96e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Bakugo123/dpo-llama3-8b-instruct-cloud-zero-with-ocr-qa-test', endpoint='https://huggingface.co', repo_type='model', repo_id='Bakugo123/dpo-llama3-8b-instruct-cloud-zero-with-ocr-qa-test'), pr_revision=None, pr_num=None)

In [ ]:
torch.cuda.empty_cache()

## Load fine tune model